In [25]:
import py2neo
from py2neo import Graph
import json

In [26]:
# Connect to the database
REMOTE_HOST = "192.168.11.174"
graph = Graph(f"bolt://{REMOTE_HOST}:7687", auth=("neo4j", "admin123"))

try:
    version = graph.run("CALL dbms.components() YIELD versions RETURN versions[0]").evaluate()
    print(f"Connected to Neo4j version: {version}")
except Exception as e:
    print(f"Connection failed: {e}")

Connected to Neo4j version: 5.26.5


In [27]:
try:
    # Check GDS
    gds_version = graph.run("RETURN gds.version()").evaluate()
    print(f"GDS Library found: Version {gds_version}")
except Exception as e:
    print("GDS is NOT accessible")
    print(f"Error details: {e}")

GDS Library found: Version 2.13.4


In [28]:
# ==========================================
# CONFIGURATION
# ==========================================
GRAPH_NAME = "algo_projection"
NODE_LABEL = "Entity"
dimensions = [32, 64, 128]
# REL_TYPES = [
#     "LOCATION_OF",
#     "PROCESS_OF",
#     "ISA",
#     "CAUSES",
#     "AFFECTS",
#     "COEXISTS_WITH",
#     "TREATS",
#     "DISRUPTS",
#     "PART_OF",
#     "ASSOCIATED_WITH",
#     "METHOD_OF",
#     "INHIBITS",
#     "DIAGNOSES"
# ]

# function to print bold headers
def print_header(title):
    print(f"\n{'='*60}\n {title}\n{'='*60}")

print_header("Fetching Relationship Types")
# Dynamically ask Neo4j for EVERY relationship type in the database!
# This guarantees we miss absolutely nothing (STIMULATES, AFFECTS, etc.)
try:
    rel_types_result = graph.run("CALL db.relationshipTypes() YIELD relationshipType RETURN collect(relationshipType) AS rels").data()
    REL_TYPES = rel_types_result[0]['rels']
    print(f"Found {len(REL_TYPES)} relationship types in the database:")
    print(REL_TYPES)
except Exception as e:
    print(f"Failed to fetch relationship types: {e}")
    raise e

# Build the configuration dictionary dynamically using the fetched list
rel_config = {
    rel_name: {"orientation": "NATURAL"} 
    for rel_name in REL_TYPES
}


 Fetching Relationship Types
Found 32 relationship types in the database:
['MEASURES', 'CAUSES', 'COEXISTS_WITH', 'ADMINISTERED_TO', 'PREVENTS', 'PROCESS_OF', 'AFFECTS', 'DISRUPTS', 'LOCATION_OF', 'TREATS', 'INTERACTS_WITH', 'PRODUCES', 'AUGMENTS', 'STIMULATES', 'METHOD_OF', 'PREDISPOSES', 'USES', 'ISA', 'PART_OF', 'ASSOCIATED_WITH', 'DIAGNOSES', 'compared_with', 'INHIBITS', 'CONVERTS_TO', 'higher_than', 'OCCURS_IN', 'same_as', 'PRECEDES', 'COMPLICATES', 'lower_than', 'MANIFESTATION_OF', 'MENTIONED_IN']


In [29]:
print_header("Creating Graph Projection")

# Drop existing projection
try:
    graph.run(f"CALL gds.graph.drop('{GRAPH_NAME}', false)")
    print(f"Dropped old projection '{GRAPH_NAME}'.")
except: pass

rel_config = {
    rel_name: {"orientation": "NATURAL"} 
    for rel_name in REL_TYPES
}



 Creating Graph Projection
Dropped old projection 'algo_projection'.


In [30]:
# Project the graph into memory
projection_query = """
CALL gds.graph.project(
    $graph_name,
    $node_label,
    $rel_config
)
"""

try:
    result = graph.run(
        projection_query, 
        graph_name=GRAPH_NAME, 
        node_label=NODE_LABEL, 
        rel_config=rel_config
    ).data()

    node_count = result[0]['nodeCount']
    rel_count = result[0]['relationshipCount']
    print(f"Graph '{GRAPH_NAME}' created in memory.")
    print(f"   Nodes (Entities): {node_count}")
    print(f"   Relationships (Semantic): {rel_count}")
    
    if rel_count == 0:
        print("WARNING: 0 relationships found. Check your filters.")
        
except Exception as e:
    print(f"Projection failed: {e}")
    raise e

Graph 'algo_projection' created in memory.
   Nodes (Entities): 33724
   Relationships (Semantic): 52915


In [31]:
# function to execute queries
def run_query(query, params={}):
    try:
        graph.run(query, params)
        print("Query Executed Successfully")
    except Exception as e:
        print(f"Error: {e}")
        raise e

# ==========================================
# Centrality Algorithms (Importance)
# ==========================================
print_header("Running Centrality Algorithms")

# ------------------------------------------
# HITS (SKIPPED)
# ------------------------------------------
# HITS requires a DIRECTED graph. 
# It calculates two scores:
#  - authScore: "How much do others point to me?" (Good for Diseases)
#  - hubScore:  "How much do I point to others?" (Good for Drugs)

print("Running HITS...")
hits_query = f"""
CALL gds.hits.write('{GRAPH_NAME}', {{
    hitsIterations: 20,
    authProperty: 'authScore',  // Writes the Authority score
    hubProperty: 'hubScore'     // Writes the Hub score
}})
"""
run_query(hits_query)

# ------------------------------------------
# PageRank (Global Importance)
# ------------------------------------------
# This ranks nodes by how many other important nodes point to them.
# GDS PageRank handles undirected graphs automatically.
print("Running PageRank...")
pagerank_query = f"""
CALL gds.pageRank.write('{GRAPH_NAME}', {{
    maxIterations: 20,
    dampingFactor: 0.85,
    writeProperty: 'pageRank'
}})
"""
run_query(pagerank_query)


 Running Centrality Algorithms
Running HITS...
Query Executed Successfully
Running PageRank...
Query Executed Successfully


In [32]:
# ==========================================
# Community Detection (Grouping)
# ==========================================
print_header("Running Community Detection")

# Louvain (Finds clusters/communities)
# This algorithm assigns a 'communityId' integer to every node.
# Nodes with the same ID belong to the same dense cluster.
print("Running Louvain...")

louvain_query = f"""
CALL gds.louvain.write('{GRAPH_NAME}', {{
    writeProperty: 'communityId'
}})
"""
run_query(louvain_query)

# Check how many communities were found
try:
    stats = graph.run(f"CALL gds.louvain.stats('{GRAPH_NAME}') YIELD communityCount").evaluate()
    print(f" Louvain finished. Found {stats} distinct communities.")
except Exception as e:
    print(f"Could not fetch stats (but algorithm likely ran): {e}")


 Running Community Detection
Running Louvain...
Query Executed Successfully
 Louvain finished. Found 23425 distinct communities.


In [33]:
# ==========================================
# Running Graph Embeddings
# ==========================================
print_header("Running Graph Embeddings")

# -------------------------------------------------------------------------
# REFRESH THE GRAPH
# must reload the graph to include the new 'pageRank' and 'communityId'
# -------------------------------------------------------------------------
print("Refreshing In-Memory Graph to include new properties...")

# Drop the old projection
try:
    graph.run(f"CALL gds.graph.drop('{GRAPH_NAME}', false)")
except: pass

# Re-project with Node Properties
projection_query = """
CALL gds.graph.project(
    $graph_name,
    $node_label,
    $rel_config,
    {
        nodeProperties: ['pageRank', 'communityId']
    }
)
"""

# Re-use existing configuration variables
graph.run(
    projection_query, 
    graph_name=GRAPH_NAME, 
    node_label=NODE_LABEL, 
    rel_config=rel_config
)
print("Graph refreshed! Properties 'pageRank' and 'communityId' are now loaded.")


# -------------------------------------------------------------------------
# GENERATE EMBEDDINGS IN A LOOP
# -------------------------------------------------------------------------
for dim in dimensions:
    print("-" * 50)
    print(f"GENERATING {dim}-DIMENSIONAL EMBEDDINGS")
    print("-" * 50)

    # ------------------------------------------
    # FastRP (Fast Random Projection)
    # ------------------------------------------
    print("Running FastRP ({dim})...")
    fastrp_query = f"""
    CALL gds.fastRP.write('{GRAPH_NAME}', {{
        embeddingDimension: {dim},
        normalizationStrength: 0.5,
        writeProperty: 'fastRP{dim}_embedding'
    }})
    """
    run_query(fastrp_query)
    
    # ------------------------------------------
    # Node2Vec (Random Walk based)
    # ------------------------------------------
    print("Running Node2Vec ({dim})...")
    node2vec_query = f"""
    CALL gds.node2vec.write('{GRAPH_NAME}', {{
        embeddingDimension: {dim},
        walkLength: 80,
        iterations: 1,
        writeProperty: 'node2vec{dim}_embedding'
    }})
    """
    run_query(node2vec_query)
    
    # ------------------------------------------
    # GraphSAGE 
    # ------------------------------------------
    print("Running GraphSAGE ({dim})...")
    model_name = f'mySAGEModel_{dim}'
    
    # Train the model
    print("   ...Training Model")
    try:
        graph.run("CALL gds.beta.model.drop('{model_name}')") 
    except: pass
    
    train_sage = f"""
    CALL gds.beta.graphSage.train('{GRAPH_NAME}', {{
        modelName: '{model_name}',
        featureProperties: ['pageRank', 'communityId'], 
        embeddingDimension: {dim},
        epochs: 10
    }})
    """
    run_query(train_sage)
    
    # Write the embeddings
    print("   ...Writing Embeddings")
    write_sage = f"""
    CALL gds.beta.graphSage.write('{GRAPH_NAME}', {{
        modelName: '{model_name}',
        writeProperty: 'graphSage{dim}_embedding'
    }})
    """
    run_query(write_sage)
    
    # ------------------------------------------
    # HashGNN
    # ------------------------------------------
    print("Running HashGNN ({dim})...")
    
    # Calculate the integer number of active bits (features)
    active_features_count = int(dim / 2)
    
    # Mutate (Calculate in memory)
    hashgnn_mutate_query = f"""
    CALL gds.hashgnn.mutate('{GRAPH_NAME}', {{
        mutateProperty: 'hashgnn{dim}_embedding', 
        embeddingDensity: {dim},
        iterations: 2,
        
        generateFeatures: {{
            dimension: {dim},
            densityLevel: {active_features_count} 
        }},
        
        randomSeed: 42
    }})
    """
    run_query(hashgnn_mutate_query)
    
    # Write to Disk (Save to Database)
    print("   ...Write HashGNN to Database")
    write_to_disk_query = f"""
    CALL gds.graph.writeNodeProperties(
        '{GRAPH_NAME}', 
        ['hashgnn{dim}_embedding']
    )
    """
    run_query(write_to_disk_query)


 Running Graph Embeddings
Refreshing In-Memory Graph to include new properties...
Graph refreshed! Properties 'pageRank' and 'communityId' are now loaded.
--------------------------------------------------
GENERATING 32-DIMENSIONAL EMBEDDINGS
--------------------------------------------------
Running FastRP ({dim})...
Query Executed Successfully
Running Node2Vec ({dim})...
Query Executed Successfully
Running GraphSAGE ({dim})...
   ...Training Model
Query Executed Successfully
   ...Writing Embeddings
Query Executed Successfully
Running HashGNN ({dim})...
Query Executed Successfully
   ...Write HashGNN to Database
Query Executed Successfully
--------------------------------------------------
GENERATING 64-DIMENSIONAL EMBEDDINGS
--------------------------------------------------
Running FastRP ({dim})...
Query Executed Successfully
Running Node2Vec ({dim})...
Query Executed Successfully
Running GraphSAGE ({dim})...
   ...Training Model
Query Executed Successfully
   ...Writing Embeddin

In [34]:
# ==========================================
# Verification & Cleanup
# ==========================================
print_header("Verification")
query_verify = f"""
MATCH (n:{NODE_LABEL}) 
WHERE n.fastRP128_embedding IS NOT NULL 
RETURN 
    n.id AS Name, 
    n.communityId AS Community, 
    n.pageRank AS PageRank,
    n.authScore AS AuthScore,
    n.hubScore AS HubScore,
    size(n.node2vec32_embedding) AS Node2Vec_32_Size,
    size(n.node2vec64_embedding) AS Node2Vec_64_Size,
    size(n.node2vec128_embedding) AS Node2Vec_128_Size,
    size(n.fastRP128_embedding) AS FastRP_128_Size
LIMIT 3
"""

try:
    results = graph.run(query_verify).data()
    print("SUCCESS! Sample of embedded data:\n")
    for i, row in enumerate(results):
        print(f"--- Node {i+1}: {row.get('Name', 'Unknown')} ---")
        print(f"Community ID    : {row.get('Community')}")
        print(f"PageRank        : {float(row.get('PageRank') or 0):.6f}")
        
        # safely handle None values for formatting
        auth = float(row.get('AuthScore') or 0)
        hub = float(row.get('HubScore') or 0)
        print(f"HITS Authority  : {auth:.6f}")
        print(f"HITS HubScore   : {hub:.6f}")
        
        print("Vector Sizes:")
        print(f"  - Node2Vec : 32-dim={row.get('Node2Vec_32_Size')}, 64-dim={row.get('Node2Vec_64_Size')}, 128-dim={row.get('Node2Vec_128_Size')}")
        print(f"  - FastRP   : 128-dim={row.get('FastRP_128_Size')}")
        print("-" * 40)

except Exception as e:
    print(f"Verification failed: {e}")

# Cleanup 
# We now need to drop the projection AND all 3 GraphSAGE models
print("\n...Cleaning up memory...")
try:
    graph.run(f"CALL gds.graph.drop('{GRAPH_NAME}', false)")
    print("In-memory graph dropped.")
    
    # Drop all 3 sizes of the GraphSAGE models
    for dim in [32, 64, 128]:
        try:
            graph.run(f"CALL gds.beta.model.drop('mySAGEModel_{dim}')")
        except:
            pass
    print("In-memory models dropped.")
except Exception as e:
    print(f"Cleanup note: {e}")


 Verification
SUCCESS! Sample of embedded data:

--- Node 1: C1293116 ---
Community ID    : 17421
PageRank        : 0.518506
HITS Authority  : 0.002028
HITS HubScore   : 0.002028
Vector Sizes:
  - Node2Vec : 32-dim=32, 64-dim=64, 128-dim=128
  - FastRP   : 128-dim=128
----------------------------------------
--- Node 2: C1882115 ---
Community ID    : 13923
PageRank        : 0.154962
HITS Authority  : 0.000014
HITS HubScore   : 0.000014
Vector Sizes:
  - Node2Vec : 32-dim=32, 64-dim=64, 128-dim=128
  - FastRP   : 128-dim=128
----------------------------------------
--- Node 3: C0205124 ---
Community ID    : 17421
PageRank        : 0.150000
HITS Authority  : 0.000000
HITS HubScore   : 0.000000
Vector Sizes:
  - Node2Vec : 32-dim=32, 64-dim=64, 128-dim=128
  - FastRP   : 128-dim=128
----------------------------------------

...Cleaning up memory...
In-memory graph dropped.
In-memory models dropped.


In [35]:
# INSPECT VECTORS
print_header("INSPECTING VECTOR DATA")

# Query to get the actual lists of numbers
# We slice the array in Cypher [0..5] to just get the start
query_inspect = f"""
MATCH (n:{NODE_LABEL})
WHERE n.fastRP128_embedding IS NOT NULL
RETURN 
    n.id AS Name,
    n.authScore AS AuthScore,
    n.hubScore AS HubScore,
    n.fastRP128_embedding[0..5] AS FastRP_Preview,
    n.node2vec128_embedding[0..5] AS Node2Vec_Preview,
    n.graphSage128_embedding[0..5] AS GraphSAGE_Preview,
    n.hashgnn128_embedding[0..5] AS HashGNN_Preview
LIMIT 5
"""

try:
    results = graph.run(query_inspect).data()
    
    for i, row in enumerate(results):
        print(f"\nNODE {i+1}: {row.get('Name', 'Unnamed Entity')}")

        auth = float(row.get('AuthScore') or 0.0)
        hub = float(row.get('HubScore') or 0.0)
        print(f"   HITS Scores: Authority={auth:.6f} | Hub={hub:.6f}")
        print("   " + "-"*30)
        
        # Updated labels to reflect that we are looking at the 128-dim vectors
        print("   FastRP 128 (first 5 values):")
        print(f"   {row['FastRP_Preview']} ...")
        
        print("   Node2Vec 128 (first 5 values):")
        print(f"   {row['Node2Vec_Preview']} ...")
        
        print("   GraphSAGE 128 (first 5 values):")
        print(f"   {row['GraphSAGE_Preview']} ...")
        
        print("   HashGNN 128 (first 5 values):")
        print(f"   {row['HashGNN_Preview']} ...")
        print("-" * 50)
        
except Exception as e:
    print(f"Could not inspect vectors: {e}")


 INSPECTING VECTOR DATA

NODE 1: C1293116
   HITS Scores: Authority=0.002028 | Hub=0.002028
   ------------------------------
   FastRP 128 (first 5 values):
   [-0.38252490758895874, 0.04146251827478409, -0.21439969539642334, -0.013350671157240868, 0.18168920278549194] ...
   Node2Vec 128 (first 5 values):
   [-0.06507048010826111, 0.008847828954458237, -0.16904783248901367, -0.0741315484046936, -0.1943216621875763] ...
   GraphSAGE 128 (first 5 values):
   [0.05505662570185379, 0.011790491377295991, 0.036089768996886915, 0.009966308183440133, 0.0075772769552139305] ...
   HashGNN 128 (first 5 values):
   [1.0, 0.0, 0.0, 1.0, 0.0] ...
--------------------------------------------------

NODE 2: C1882115
   HITS Scores: Authority=0.000014 | Hub=0.000014
   ------------------------------
   FastRP 128 (first 5 values):
   [-0.4164165258407593, -0.024912554770708084, -0.14457391202449799, -0.004683703184127808, 0.04286130145192146] ...
   Node2Vec 128 (first 5 values):
   [0.000571218784

FastRP gets 0s because these nodes has no connections (we removed the relationships between entities and articles) and it calculates a node's embedding based on its neighbors.
Self Influence?